In [ ]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
import kagglehub
path = kagglehub.dataset_download("kmader/food41")
print("Датасет загружен в:", path)

Using Colab cache for faster access to the 'food41' dataset.
Датасет загружен в: /kaggle/input/food41


In [ ]:
# ================================
# 1. УСТАНОВКА И НАСТРОЙКИ
# ================================
!pip install -q kagglehub torch torchvision tqdm matplotlib scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import copy
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import kagglehub

# --- Скачивание датасета (один раз) ---
DATASET_PATH = kagglehub.dataset_download("kmader/food41")
print("Датасет загружен в:", DATASET_PATH)

# --- Проверка GPU ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {DEVICE}")

# --- Параметры ---
IMAGES_DIR = os.path.join(DATASET_PATH, "images")
META_DIR = os.path.join(DATASET_PATH, "meta", "meta")
TRAIN_FILE = os.path.join(META_DIR, "train.txt")
TEST_FILE = os.path.join(META_DIR, "test.txt")
CLASSES_FILE = os.path.join(META_DIR, "classes.txt")
MODEL_SAVE_PATH = "best_model_food41.pth"
CLASSES_OUT = "food101_classes.txt"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 64                     # GPU позволяет больше
NUM_EPOCHS_HEAD = 10
NUM_EPOCHS_FINETUNE = 20
LEARNING_RATE_HEAD = 0.001
LEARNING_RATE_FINETUNE = 0.0001
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
IMAGE_SIZE = 300                    # для EfficientNet-B3
NUM_WORKERS = 2                     # для Colab нормально
CONFIDENCE_THRESHOLD = 0.75

# ================================
# 2. DATASET (формат apple_pie/1357950)
# ================================
class FoodDataset(Dataset):
    def __init__(self, root_images, split_file, class_to_idx, transform=None):
        self.root = root_images
        self.transform = transform
        self.samples = []

        with open(split_file, "r") as f:
            for line in f:
                rel_path_no_ext = line.strip()
                if not rel_path_no_ext:
                    continue
                rel_path = rel_path_no_ext + ".jpg"
                class_name = rel_path_no_ext.split("/")[0]
                if class_name not in class_to_idx:
                    continue
                label = class_to_idx[class_name]
                full_path = os.path.join(root_images, rel_path)

                if not os.path.exists(full_path):
                    full_path_jpeg = os.path.join(root_images, rel_path_no_ext + ".jpeg")
                    if os.path.exists(full_path_jpeg):
                        full_path = full_path_jpeg
                    else:
                        continue
                self.samples.append((full_path, label))
        print(f"Загружено {len(self.samples)} изображений из {split_file}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# ================================
# 3. ТРАНСФОРМАЦИИ
# ================================
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ================================
# 4. МОДЕЛЬ (EfficientNet-B3)
# ================================
def create_model(num_classes):
    model = models.efficientnet_b3(weights="IMAGENET1K_V1")
    for param in model.parameters():
        param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.SiLU(),
        nn.Dropout(p=0.3),
        nn.Linear(1024, num_classes)
    )
    return model

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        n_classes = pred.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * torch.log_softmax(pred, dim=1), dim=1))

# ================================
# 5. МЕТРИКИ
# ================================
def compute_metrics(all_labels, all_preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    acc = accuracy_score(all_labels, all_preds)
    return acc, precision, recall, f1

# ================================
# 6. ОБУЧЕНИЕ ОДНОЙ ЭПОХИ (с прогресс-баром)
# ================================
def train_one_epoch(model, loader, criterion, optimizer, device, phase='train'):
    if phase == 'train':
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f'{phase}', leave=False)
    with torch.set_grad_enabled(phase == 'train'):
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if phase == 'train':
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.3f}'})

    epoch_loss = running_loss / len(loader.dataset)
    acc, prec, rec, f1 = compute_metrics(all_labels, all_preds)
    return epoch_loss, acc, prec, rec, f1

# ================================
# 7. ГЛАВНЫЙ ЦИКЛ ОБУЧЕНИЯ
# ================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, start_epoch=0, history=None):
    if history is None:
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
    best_f1 = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, start_epoch + num_epochs):
        print(f'\n{"="*50}')
        print(f'Эпоха {epoch+1}/{start_epoch + num_epochs}')
        print(f'{"="*50}')

        train_loss, train_acc, train_prec, train_rec, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE, phase='train'
        )
        print(f'Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}')

        val_loss, val_acc, val_prec, val_rec, val_f1 = train_one_epoch(
            model, val_loader, criterion, optimizer, DEVICE, phase='val'
        )
        print(f'Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}')
        print(f'        Prec: {val_prec:.4f} | Recall: {val_rec:.4f}')

        if isinstance(scheduler, lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f'  ** Сохранена лучшая модель (F1: {best_f1:.4f}) **')

        if (epoch + 1) % 3 == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_f1': best_f1,
                'history': history
            }
            torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth'))

    model.load_state_dict(best_model_wts)
    return model, history

# ================================
# 8. ЗАПУСК ОБУЧЕНИЯ
# ================================
if __name__ == '__main__':
    # Загрузка классов
    with open(CLASSES_FILE, 'r') as f:
        class_names = [line.strip() for line in f.readlines()]
    num_classes = len(class_names)
    print(f"Найдено классов: {num_classes}")

    with open(CLASSES_OUT, 'w') as f:
        for cls in class_names:
            f.write(cls + '\n')

    class_to_idx = {cls: i for i, cls in enumerate(class_names)}

    train_dataset = FoodDataset(IMAGES_DIR, TRAIN_FILE, class_to_idx, train_transforms)
    val_dataset   = FoodDataset(IMAGES_DIR, TEST_FILE,  class_to_idx, val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Размеры: Train={len(train_dataset)}, Val={len(val_dataset)}")

    model = create_model(num_classes).to(DEVICE)
    criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTHING)

    # ЭТАП 1: обучение верхушки
    print("\n" + "="*60)
    print("ЭТАП 1: Обучение классификатора (features заморожены)")
    print("="*60)
    optimizer_head = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=LEARNING_RATE_HEAD, weight_decay=WEIGHT_DECAY)
    scheduler_head = lr_scheduler.CosineAnnealingLR(optimizer_head, T_max=NUM_EPOCHS_HEAD)
    model, history_head = train_model(
        model, train_loader, val_loader, criterion, optimizer_head, scheduler_head,
        NUM_EPOCHS_HEAD
    )

    # ЭТАП 2: тонкая настройка
    print("\n" + "="*60)
    print("ЭТАП 2: Тонкая настройка (разморозка части features)")
    print("="*60)
    # Для B3 размораживаем блоки с 5-го и дальше (всего 7 блоков)
    unlock_from = 5
    for name, param in model.named_parameters():
        if 'features' in name:
            parts = name.split('.')
            if parts[1].isdigit():
                block_num = int(parts[1])
                param.requires_grad = (block_num >= unlock_from)
        else:
            param.requires_grad = True

    classifier_params = list(model.classifier.parameters())
    feature_params = [p for n, p in model.named_parameters() if 'features' in n and p.requires_grad]
    optimizer_finetune = optim.AdamW([
        {'params': classifier_params, 'lr': LEARNING_RATE_FINETUNE},
        {'params': feature_params, 'lr': LEARNING_RATE_FINETUNE / 10}
    ], weight_decay=WEIGHT_DECAY)
    scheduler_finetune = lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer_finetune, T_0=5, T_mult=2, eta_min=1e-6
    )
    model, history_finetune = train_model(
        model, train_loader, val_loader, criterion, optimizer_finetune, scheduler_finetune,
        NUM_EPOCHS_FINETUNE
    )

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"\nИтоговая модель сохранена в {MODEL_SAVE_PATH}")

    # Графики
    full_history = {
        'train_loss': history_head['train_loss'] + history_finetune['train_loss'],
        'val_loss': history_head['val_loss'] + history_finetune['val_loss'],
        'train_acc': history_head['train_acc'] + history_finetune['train_acc'],
        'val_acc': history_head['val_acc'] + history_finetune['val_acc']
    }
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.plot(full_history['train_loss'], label='Train Loss')
    plt.plot(full_history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)

    plt.subplot(1,2,2)
    plt.plot(full_history['train_acc'], label='Train Accuracy')
    plt.plot(full_history['val_acc'], label='Val Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Готово!")

Using Colab cache for faster access to the 'food41' dataset.
Датасет загружен в: /kaggle/input/food41
Используемое устройство: cuda
Найдено классов: 101
Загружено 75750 изображений из /kaggle/input/food41/meta/meta/train.txt
Загружено 25250 изображений из /kaggle/input/food41/meta/meta/test.txt
Размеры: Train=75750, Val=25250
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 175MB/s]



ЭТАП 1: Обучение классификатора (features заморожены)

Эпоха 1/10


Train -> Loss: 3.0914 | Acc: 0.3470 | F1: 0.3367


Val   -> Loss: 2.2472 | Acc: 0.5791 | F1: 0.5693
        Prec: 0.5935 | Recall: 0.5791
  ** Сохранена лучшая модель (F1: 0.5693) **

Эпоха 2/10


...

Val   -> Loss: 2.0592 | Acc: 0.6649 | F1: 0.6597
        Prec: 0.6689 | Recall: 0.6649

ЭТАП 2: Тонкая настройка (разморозка части features)

Эпоха 1/20


Train -> Loss: 2.3747 | Acc: 0.5439 | F1: 0.5385


Val   -> Loss: 1.8749 | Acc: 0.7271 | F1: 0.7240
        Prec: 0.7313 | Recall: 0.7271
  ** Сохранена лучшая модель (F1: 0.7240) **

Эпоха 2/20


Train -> Loss: 2.1909 | Acc: 0.5990 | F1: 0.5953


val:  70%|███████   | 278/395 [02:34<01:01,  1.89it/s, loss=1.473]

In [ ]:
from google.colab import files
files.download('best_model_food41.pth')
files.download('food101_classes.txt')